In [25]:
import pandas as pd
import numpy as np
import os
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

In [26]:
# Load dataset from root folder (has upazila column)
df = pd.read_csv("../smartgrid_risk_dataset.csv")
print(df.shape)
df.head()

(15000, 27)


,datetime,year,month,day,hour,weekday,temperature,humidity,rainfall,wind_speed,...,upazila,area_type,substation_id,feeder_id,transformer_age,transformer_capacity,outage_history,maintenance_due,population_density,industrial_load_ratio
0,2026-04-13 00:00:00,2026,4,13,0,0,24.62,81.61,0.0,4.78,...,Bahubal,Rural,SS_033,FDR_20,11,200,3,No,1966,0.19
1,2026-04-13 02:00:00,2026,4,13,2,0,23.38,82.84,0.0,4.92,...,Chhatak,Rural,SS_007,FDR_12,7,300,6,No,588,0.22
2,2026-04-13 04:00:00,2026,4,13,4,0,22.85,86.41,0.0,4.82,...,Chunarughat,Rural,SS_015,FDR_04,14,350,1,Yes,1801,0.25
3,2026-04-13 06:00:00,2026,4,13,6,0,23.05,87.09,0.0,5.03,...,Golapganj,Rural,SS_045,FDR_18,13,250,10,No,614,0.07
4,2026-04-13 08:00:00,2026,4,13,8,0,24.15,86.65,0.0,4.79,...,Bishwanath,Rural,SS_016,FDR_18,13,400,2,Yes,1697,0.13


In [27]:
print(df.isnull().sum())

datetime                 0
year                     0
month                    0
day                      0
hour                     0
weekday                  0
temperature              0
humidity                 0
rainfall                 0
wind_speed               0
weather_state            0
electricity_demand       0
renewable_generation     0
transformer_load         0
risk_score               0
risk_level               0
district                 0
upazila                  0
area_type                0
substation_id            0
feeder_id                0
transformer_age          0
transformer_capacity     0
outage_history           0
maintenance_due          0
population_density       0
industrial_load_ratio    0
dtype: int64


In [28]:
print(df["risk_level"].value_counts())
print(df["risk_level"].value_counts(normalize=True))

risk_level
Medium    7427
High      3992
Low       3581
Name: count, dtype: int64
risk_level
Medium    0.495133
High      0.266133
Low       0.238733
Name: proportion, dtype: float64


In [29]:
# Drop datetime, risk_score, and date features (year, month, day)
df = df.drop(columns=[
    "datetime",
    "risk_score",
    "year",
    "month",
    "day"
])

print("After dropping columns:", df.shape)

After dropping columns: (15000, 22)


In [30]:
categorical_cols = [
    "weather_state",
    "district",
    "upazila",
    "area_type",
    "substation_id",
    "feeder_id",
    "maintenance_due"
]

encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = le

In [31]:
target_encoder = LabelEncoder()
df["risk_level"] = target_encoder.fit_transform(df["risk_level"])

In [32]:
X = df.drop("risk_level", axis=1)
y = df["risk_level"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [33]:

os.makedirs("../dataset", exist_ok=True)

# Save unscaled data for Decision Tree and Random Forest
np.save("../dataset/X_train.npy", X_train)
np.save("../dataset/X_test.npy", X_test)
np.save("../dataset/y_train.npy", y_train)
np.save("../dataset/y_test.npy", y_test)

print("Train/Test datasets saved successfully.")

Train/Test datasets saved successfully.


In [34]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training Shape :", X_train_scaled.shape)
print("Testing Shape  :", X_test_scaled.shape)

Training Shape : (12000, 21)
Testing Shape  : (3000, 21)


In [35]:


os.makedirs("../models", exist_ok=True)
os.makedirs("../dataset", exist_ok=True)

joblib.dump(scaler, "../models/scaler.pkl")
joblib.dump(target_encoder, "../models/target_encoder.pkl")

# Save scaled data
np.save("../dataset/X_train_scaled.npy", X_train_scaled)
np.save("../dataset/X_test_scaled.npy", X_test_scaled)

print("Saved successfully!")

Saved successfully!


In [36]:
print("Feature columns:")
print(list(X.columns))

Feature columns:
['hour', 'weekday', 'temperature', 'humidity', 'rainfall', 'wind_speed', 'weather_state', 'electricity_demand', 'renewable_generation', 'transformer_load', 'district', 'upazila', 'area_type', 'substation_id', 'feeder_id', 'transformer_age', 'transformer_capacity', 'outage_history', 'maintenance_due', 'population_density', 'industrial_load_ratio']
